<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        Exploratory Data Analysis for EHR Mortality Risk Prediction
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To explore hospital-level and patient-level patterns in Electronic Health Records (EHR) and identify key signals associated with in-hospital mortality.

### **1. Setup and Imports**

In [29]:
# Core libraries for data handling
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 200)

print("Libraries imported successfully.")

Libraries imported successfully.


### **2. Load the Processed Master Table**

In [30]:
# Load the processed master table
df = pd.read_parquet("../data/processed/master_table.parquet")

print("Master table loaded successfully.")
print(f"Shape: {df.shape}")

Master table loaded successfully.
Shape: (546028, 46)


In [31]:
df["admission_type"].value_counts()

admission_type
other       477969
urgent       54929
elective     13130
Name: count, dtype: int64

### **3. Preview the Dataset**

In [32]:
# Display the first few rows
df.head()

,subject_id,hadm_id,admittime,age,gender,admission_type,marital_status,race,ed_los_hours,came_from_ed,admission_hour,admission_day_of_week,num_prev_visits,time_since_last_visit_days,avg_time_between_visits_days,avg_prev_los,max_prev_los,last_los,avg_prev_ed_los,max_prev_ed_los,num_prev_diagnoses,num_unique_diagnoses_icd_codes,num_prev_procedures,num_unique_procedure_icd_codes,avg_prev_drg_severity,max_prev_drg_severity,avg_prev_drg_mortality,max_prev_drg_mortality,num_abnormal_labs,abnormal_lab_ratio,stat_lab_ratio,last_lab_flag,last_norm_lab,num_prev_medications,num_unique_drugs,num_prev_transfers,num_prev_icu_visits,num_prev_infections,num_unique_organisms,num_resistant_cases,bmi_last,weight_last,last_systolic,last_diastolic,last_pulse_pressure,target
0,10008064,22898614,2180-04-05 08:00:00,44.0,0.0,other,married,black,0.00,0,8,2,0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.0,0.0,0.0,0.0,15,0.16,0.00,0,NaN,0,0,1,0,1,0,0,48.4,328.0,140.0,93.0,47.0,0
1,10008064,20742244,2185-04-18 15:32:00,49.0,0.0,other,married,black,23.67,1,15,0,1,1839.31,0.0,1.25,1.25,1.25,0.00,0.00,7,7,1,1,1.0,1.0,1.0,1.0,28,0.14,0.13,0,NaN,17,10,4,0,1,0,0,44.8,303.6,132.0,100.0,32.0,0
2,10008960,29069456,2118-10-22 14:26:00,65.0,1.0,other,single,white,6.33,1,14,5,0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.0,0.0,0.0,0.0,22,0.24,0.99,0,NaN,0,0,1,0,3,0,0,NaN,NaN,NaN,NaN,NaN,0
3,10008960,29075733,2118-11-19 02:49:00,65.0,1.0,other,single,white,16.62,1,2,5,1,27.52,0.0,26.21,26.21,26.21,6.33,6.33,11,11,0,0,3.0,3.0,3.0,3.0,188,0.25,0.20,0,NaN,141,33,5,0,3,0,0,NaN,227.7,NaN,NaN,NaN,0
4,10024704,26208346,2182-03-19 22:08:00,30.0,0.0,urgent,single,hispanic_latino,0.00,0,22,1,0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.0,0.0,0.0,0.0,9,0.47,1.00,0,NaN,0,0,0,0,2,0,0,NaN,NaN,NaN,NaN,NaN,0


### **4. Inspect Columns and Data Types**

In [33]:
# Show all columns and their data types
column_info = pd.DataFrame({
    "column_name": df.columns,
    "data_type": df.dtypes.astype(str).values
})

column_info

,column_name,data_type
0,subject_id,int64
1,hadm_id,int64
2,admittime,datetime64[ns]
3,age,float64
4,gender,float64
5,admission_type,str
6,marital_status,str
7,race,str
8,ed_los_hours,float64
9,came_from_ed,int64


### **5. Basic Dataset Summary**

In [34]:
# General high-level summary of the dataset
num_rows, num_cols = df.shape
num_patients = df["subject_id"].nunique()
num_admissions = df["hadm_id"].nunique()

print("Dataset Summary:")
print(f"Number of rows (admissions): {num_rows:,}")
print(f"Number of columns: {num_cols:,}")
print(f"Number of unique patients: {num_patients:,}")
print(f"Number of unique admissions: {num_admissions:,}")

Dataset Summary:
Number of rows (admissions): 546,028
Number of columns: 46
Number of unique patients: 223,452
Number of unique admissions: 546,028


### **6. Missing Values Summary**

In [35]:
# Compute missing values per column
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().sum() / len(df) * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
last_norm_lab,297530,54.49
bmi_last,269545,49.36
last_pulse_pressure,241595,44.25
last_diastolic,241595,44.25
last_systolic,241595,44.25
weight_last,239165,43.80
max_prev_drg_severity,45981,8.42
avg_prev_drg_mortality,45981,8.42
max_prev_drg_mortality,45981,8.42
avg_prev_drg_severity,45981,8.42


In [36]:
# Filter to only columns that contain missing values
missing_only = missing_summary[missing_summary["missing_count"] > 0]

missing_only

,missing_count,missing_percent
last_norm_lab,297530,54.49
bmi_last,269545,49.36
last_pulse_pressure,241595,44.25
last_diastolic,241595,44.25
last_systolic,241595,44.25
weight_last,239165,43.80
max_prev_drg_severity,45981,8.42
avg_prev_drg_mortality,45981,8.42
max_prev_drg_mortality,45981,8.42
avg_prev_drg_severity,45981,8.42


### **8. Duplicate Checks**

In [37]:
# Check for duplicate full rows
duplicate_rows = df.duplicated().sum()

# Check whether the admission-level key is unique
duplicate_keys = df.duplicated(subset=["subject_id", "hadm_id"]).sum()

print("Duplicate Checks:")
print(f"Duplicate full rows: {duplicate_rows}")
print(f"Duplicate (subject_id, hadm_id) pairs: {duplicate_keys}")

Duplicate Checks:
Duplicate full rows: 0
Duplicate (subject_id, hadm_id) pairs: 0


### **9. Target Variable Summary**

In [38]:
# Inspect the binary target distribution
target_counts = df["target"].value_counts(dropna=False).sort_index()
target_percent = (df["target"].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_percent
})

target_summary.index = ["Survived (0)", "Died (1)"][:len(target_summary.index)]
target_summary

,count,percent
Survived (0),534227,97.84
Died (1),11801,2.16


### **10. Summary Statistics for Numeric Features**

In [39]:
# Summary statistics for all numeric columns
numeric_summary = df.describe().T

numeric_summary

,count,mean,min,25%,50%,75%,max,std
subject_id,546028.0,15011181.256086,10000032.0,12523805.0,15019607.5,17504026.0,19999987.0,2877694.299379
hadm_id,546028.0,25001002.583349,20000019.0,22496622.25,25003849.0,27502819.75,29999935.0,2888710.200538
admittime,546028,2155-02-18 15:33:10.993173504,2105-10-04 17:26:00,2135-02-14 13:13:30,2155-01-11 17:56:00,2175-03-15 15:55:30,2214-12-15 19:11:00,NaN
age,546028.0,59.156538,18.0,45.0,61.0,74.0,106.0,19.146315
gender,546028.0,0.479703,0.0,0.0,0.0,1.0,1.0,0.499588
ed_los_hours,546028.0,7.556426,-18.73,0.0,5.52,9.87,305.98,9.788171
came_from_ed,546028.0,0.694543,0.0,0.0,1.0,1.0,1.0,0.460601
admission_hour,546028.0,13.009791,0.0,7.0,15.0,19.0,23.0,7.32867
admission_day_of_week,546028.0,3.008392,0.0,1.0,3.0,5.0,6.0,2.001633
num_prev_visits,546028.0,3.336045,0.0,0.0,1.0,3.0,237.0,7.995328


### **11. Summary of Categorical Features**

In [40]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['admission_type', 'marital_status', 'race']


### **12. Unique Value Counts for Categorical Features**

In [41]:
# Count how many unique categories each categorical feature has
categorical_unique_summary = pd.DataFrame({
    "num_unique_values": [df[col].nunique(dropna=True) for col in categorical_cols],
    "missing_count": [df[col].isna().sum() for col in categorical_cols],
    "missing_percent": [(df[col].isna().sum() / len(df) * 100).round(2) for col in categorical_cols]
}, index=categorical_cols).sort_values("num_unique_values", ascending=False)

categorical_unique_summary

,num_unique_values,missing_count,missing_percent
race,5,0,0.00
marital_status,4,13619,2.49
admission_type,3,0,0.00


### **13. Value Counts for Categorical Features**

In [42]:
# Display category frequencies for each categorical column
for col in categorical_cols:
    print(f"Column: {col}:")
    print(f"{df[col].value_counts(dropna=False)}\n")

Column: admission_type:
admission_type
other       477969
urgent       54929
elective     13130
Name: count, dtype: int64

Column: marital_status:
marital_status
married     229134
single      206232
widowed      56687
divorced     40356
NaN          13619
Name: count, dtype: int64

Column: race:
race
white              360519
black               89057
other               44491
hispanic_latino     32210
asian               19751
Name: count, dtype: int64



### **14. Zero Value Inspection for History-Based Features**

In [43]:
# Check how often selected history features are zero
history_features = [
    "num_prev_visits",
    "time_since_last_visit_days",
    "avg_time_between_visits_days",
    "avg_prev_los",
    "max_prev_los",
    "last_los",
    "num_prev_diagnoses",
    "num_unique_icd_codes",
    "num_abnormal_labs",
    "num_prev_medications",
    "num_unique_drugs",
    "num_prev_infections",
    "num_resistant_cases"
]

existing_history_features = [col for col in history_features if col in df.columns]

zero_summary = pd.DataFrame({
    "zero_count": [(df[col] == 0).sum() for col in existing_history_features],
    "zero_percent": [((df[col] == 0).sum() / len(df) * 100).round(2) for col in existing_history_features]
}, index=existing_history_features).sort_values("zero_percent", ascending=False)

zero_summary

,zero_count,zero_percent
num_resistant_cases,452304,82.84
avg_time_between_visits_days,323619,59.27
num_prev_diagnoses,223629,40.96
time_since_last_visit_days,223466,40.93
avg_prev_los,223465,40.93
max_prev_los,223465,40.93
last_los,223502,40.93
num_prev_visits,223452,40.92
num_prev_medications,201104,36.83
num_unique_drugs,201104,36.83
